# **Day 21**
## Class Inheritance and List Slicing

### Inheritance

In [3]:
class BaseClassName:
    pass

class DerivedClassName(BaseClassName):
    def __init__(self):
        super().__init__()

The name `BaseClassName` must be defined in a namespace accessible from the scope containing the derived class definition. In place of a base class name, other arbitrary expressions are also allowed. This can be useful, for example, when the base class is defined in another module:

```python
class DerivedClassName(modname.BaseClassName):
    pass
```

Execution of a derived class definition proceeds the same as for a base class. When the class object is constructed, the base class is remembered. This is used for resolving attribute references: if a requested attribute is not found in the class, the search proceeds to look in the base class. This rule is applied recursively if the base class itself is derived from some other class.

There’s nothing special about instantiation of derived classes: `DerivedClassName()` creates a new instance of the class. Method references are resolved as follows: the corresponding class attribute is searched, descending down the chain of base classes if necessary, and the method reference is valid if this yields a function object.

Derived classes may override methods of their base classes. Because methods have no special privileges when calling other methods of the same object, a method of a base class that calls another method defined in the same base class may end up calling a method of a derived class that overrides it. (For C++ programmers: all methods in Python are effectively `virtual`. Method calls are **resolved at runtime** based on the actual type of the object, rather than being statically determined at compile time. This is a key feature of Python's dynamic dispatch and is inherent to how the language handles method invocation.)

An overriding method in a derived class may in fact want to extend rather than simply replace the base class method of the same name. There is a simple way to call the base class method directly: just call `BaseClassName.methodname(self, arguments)`. This is occasionally useful to clients as well. (Note that this only works if the base class is accessible as `BaseClassName` in the global scope.)

Python has two built-in functions that work with inheritance:

- Use [isinstance()](https://docs.python.org/3/library/functions.html#isinstance) to check an instance’s type: `isinstance(obj, int)` will be `True` only if `obj.__class__` is [int](https://docs.python.org/3/library/functions.html#int) or some class derived from [int](https://docs.python.org/3/library/functions.html#int).

- Use [issubclass()](https://docs.python.org/3/library/functions.html#issubclass) to check class inheritance: `issubclass(bool, int)` is `True` since [bool](https://docs.python.org/3/library/functions.html#bool) is a subclass of [int](https://docs.python.org/3/library/functions.html#int). However, `issubclass(float, int)` is False since [float](https://docs.python.org/3/library/functions.html#float) is not a subclass of [int](https://docs.python.org/3/library/functions.html#int).

#### Multiple Inheritance

Python supports a form of multiple inheritance as well. A class definition with multiple base classes looks like this:

```python
class DerivedClassName(Base1, Base2, Base3):
    <statement-1>
    .
    .
    .
    <statement-N>
```
For most purposes, in the simplest cases, you can think of the search for attributes inherited from a parent class as depth-first, left-to-right, not searching twice in the same class where there is an overlap in the hierarchy. Thus, if an attribute is not found in `DerivedClassName`, it is searched for in `Base1`, then (recursively) in the base classes of `Base1`, and if it was not found there, it was searched for in `Base2`, and so on.

In fact, it is slightly more complex than that; the method resolution order changes dynamically to support cooperative calls to [super()](https://docs.python.org/3/library/functions.html#super). This approach is known in some other multiple-inheritance languages as call-next-method and is more powerful than the super call found in single-inheritance languages.

Dynamic ordering is necessary because all cases of multiple inheritance exhibit one or more diamond relationships (where at least one of the parent classes can be accessed through multiple paths from the bottommost class). For example, all classes inherit from [object](https://docs.python.org/3/library/functions.html#object), so any case of multiple inheritance provides more than one path to reach [object](https://docs.python.org/3/library/functions.html#object). To keep the base classes from being accessed more than once, the dynamic algorithm linearizes the search order in a way that preserves the left-to-right ordering specified in each class, that calls each parent only once, and that is monotonic (meaning that a class can be subclassed without affecting the precedence order of its parents). Taken together, these properties make it possible to design reliable and extensible classes with multiple inheritance. For more detail, see [The Python 2.3 Method Resolution Order](https://docs.python.org/3/howto/mro.html#python-2-3-mro).

#### Private Variables

“Private” instance variables that cannot be accessed except from inside an object don’t exist in Python. However, there is a convention that is followed by most Python code: a name prefixed with an underscore (e.g. `_spam`) should be treated as a non-public part of the API (whether it is a function, a method or a data member). It should be considered an implementation detail and subject to change without notice.

Since there is a valid use-case for class-private members (namely to avoid name clashes of names with names defined by subclasses), there is limited support for such a mechanism, called *name mangling*. Any identifier of the form `__spam` (at least two leading underscores, at most one trailing underscore) is textually replaced with `_classname__spam`, where `classname` is the current class name with leading underscore(s) stripped. This mangling is done without regard to the syntactic position of the identifier, as long as it occurs within the definition of a class.

**See also:** The [private name mangling specifications](https://docs.python.org/3/reference/expressions.html#private-name-mangling) for details and special cases.

Name mangling is helpful for letting subclasses override methods without breaking intraclass method calls. For example:

In [18]:
class Mapping:
    def __init__(self, iterable):
        self.items_list = []
        self.__update(iterable)

    def update(self, iterable):
        for item in iterable:
            self.items_list.append(item)

    __update = update   # private copy of original update() method

class MappingSubclass(Mapping):

    def update(self, keys, values):
        # provides new signature for update()
        # but does not break __init__()
        for item in zip(keys, values):
            self.items_list.append(item)

The above example would work even if `MappingSubclass` were to introduce a `__update` identifier since it is replaced with `_Mapping__update` in the `Mapping` class and `_MappingSubclass__update` in the `MappingSubclass` class respectively.

Note that the mangling rules are designed mostly to avoid accidents; it still is possible to access or modify a variable that is considered private. This can even be useful in special circumstances, such as in the debugger.

Notice that code passed to `exec()` or `eval()` does not consider the classname of the invoking class to be the current class; this is similar to the effect of the `global` statement, the effect of which is likewise restricted to code that is byte-compiled together. The same restriction applies to `getattr()`, `setattr()` and `delattr()`, as well as when referencing `__dict__` directly.

In [22]:
class Animal:
    def __init__(self):
        self.num_eyes = 2

    def breathe(self):
        print("Inhale, exhale.")

class Fish(Animal):
    def __init__(self):
        super().__init__()

    def swim(self):
        print("Moving in water.")

    def breathe(self):
        super().breathe()
        print("Doing this underwater.")

nemo = Fish()

nemo.swim()
nemo.breathe()

print(nemo.num_eyes)

Moving in water.
Inhale, exhale.
Doing this underwater.
2


### Slicing

In [24]:
piano_keys = ["a", "b", "c", "d", "e", "f", "g"]
piano_tuple = ("a", "b", "c", "d", "e", "f", "g")

print(f"1: {piano_keys}")

print(f"2: {piano_keys[2:5]}")
print(f"3: {piano_keys[2:]}")
print(f"4: {piano_keys[:5]}")

print(f"5: {piano_keys[2:5:2]}")
print(f"6: {piano_keys[::2]}")

print(f"7: {piano_keys[::-1]}")
print(f"8: {piano_keys[::-3]}")

print(f"9: {piano_tuple[2:5]}")


1: ['a', 'b', 'c', 'd', 'e', 'f', 'g']
2: ['c', 'd', 'e']
3: ['c', 'd', 'e', 'f', 'g']
4: ['a', 'b', 'c', 'd', 'e']
5: ['c', 'e']
6: ['a', 'c', 'e', 'g']
7: ['g', 'f', 'e', 'd', 'c', 'b', 'a']
8: ['g', 'd', 'a']
9: ('c', 'd', 'e')
